<a href="https://colab.research.google.com/github/AjayLohith/Naive-RAG/blob/main/Naive_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#BASIC RAG APP


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


###Imports

In [ ]:
!pip install openai chromadb python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found e

In [ ]:
import os
import dotenv
from openai import OpenAI, api_key
from  dotenv import load_dotenv
import json
import chromadb
from openai.types.responses import responses_client_event
from websockets import client

In [ ]:
from google.colab import userdata
groq_api_key=userdata.get('GROQ_API_KEY')

## Interacting with LLM

In [ ]:
from openai import OpenAI

client=OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

In [ ]:
response=client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role":"system","content":"You are a professional RAG expert in Agentic AI field"},
        {"role":"user","content":"Explainn me types of rags in 5 bullet points"}
    ],
    temperature=0
)
print(response.choices[0].message.content)

As a RAG (Retrieve, Augment, Generate) expert in the Agentic AI field, I'd be happy to break down the types of RAGs for you. Here are 5 key types:

* **Retrieve-Only RAG**: This type of RAG focuses solely on retrieving relevant information from a knowledge base or database, without any augmentation or generation capabilities. It's often used for simple question-answering tasks or data retrieval.
* **Augment-Only RAG**: In contrast, an Augment-Only RAG takes existing information and augments it with additional context, entities, or relationships, but doesn't generate new text. This type of RAG is useful for tasks like entity recognition, sentiment analysis, or data enrichment.
* **Generate-Only RAG**: A Generate-Only RAG uses a given prompt or input to generate new text, without retrieving or augmenting existing information. This type of RAG is often used for tasks like text summarization, language translation, or creative writing.
* **Retrieve-Augment RAG**: This type of RAG combines t

## Loading and chunking data from document

In [ ]:
with open("/content/drive/MyDrive/Naive RAG/_data/company_hr_policy.txt","r")as f:
  hr_doc=f.read()

with open("/content/drive/MyDrive/Naive RAG/_data/engineering_standards.txt","r")as f:
  engineering_standards=f.read()

with open("/content/drive/MyDrive/Naive RAG/_data/onboarding_guide.txt","r")as f:
  onboarding_guide=f.read()

with open("/content/drive/MyDrive/Naive RAG/_data/product_knowledge_base.txt","r")as f:
  product_knowledge=f.read()

with open("/content/drive/MyDrive/Naive RAG/_data/security_policy.txt","r")as f:
  security_policy=f.read()

print(hr_doc)
# print(engineering_standards)
# print(onboarding_guide)
# print(product_knowledge)
# print(security_policy)


NovaTech Solutions — Employee Handbook & HR Policy
Version 3.2 | Last Updated: January 2026

SECTION 1: LEAVE POLICY

Annual Leave:
All full-time employees are entitled to 24 days of paid annual leave per calendar year. Leave accrues at the rate of 2 days per month. New employees can start using accrued leave after completing 3 months of service. Unused leave up to 10 days can be carried forward to the next year. Any leave beyond 10 days will lapse on December 31st.

Sick Leave:
Employees are entitled to 12 days of sick leave per year. Sick leave for more than 3 consecutive days requires a medical certificate from a registered medical practitioner. Sick leave cannot be carried forward or encashed. In case of extended illness beyond 12 days, employees may apply for medical leave without pay, subject to HR approval.

Casual Leave:
Employees are entitled to 6 days of casual leave per year. Casual leave cannot be taken for more than 3 consecutive days. Prior approval from the reporting man

In [ ]:
print(f"HR Document lenght: {len(hr_doc)} chars")
print(f"HR Document lenght: {len(engineering_standards)} chars")

print(f"HR document words {len(hr_doc.split())}")

HR Document lenght: 7464 chars
HR Document lenght: 4941 chars
HR document words 1052


## Chunking Strategy

In [12]:
def chunk_documents(text,source_name):
  paragraph=text.strip().split("\n" "\n")
  chunks=[]

  for para in paragraph:
    para=para.strip()
    if len(para)<50:
      continue

    if para.startswith("=="):
      continue

    chunks.append({"text":para,"source":source_name})


  return chunks

hr_chunk=chunk_documents(hr_doc,"HR Policy")
engineering_standards_chunk=chunk_documents(engineering_standards,"Engineering Standards")
print(hr_chunk)
print(engineering_standards_chunk)



[{'text': 'NovaTech Solutions — Employee Handbook & HR Policy\nVersion 3.2 | Last Updated: January 2026', 'source': 'HR Policy'}, {'text': 'Annual Leave:\nAll full-time employees are entitled to 24 days of paid annual leave per calendar year. Leave accrues at the rate of 2 days per month. New employees can start using accrued leave after completing 3 months of service. Unused leave up to 10 days can be carried forward to the next year. Any leave beyond 10 days will lapse on December 31st.', 'source': 'HR Policy'}, {'text': 'Sick Leave:\nEmployees are entitled to 12 days of sick leave per year. Sick leave for more than 3 consecutive days requires a medical certificate from a registered medical practitioner. Sick leave cannot be carried forward or encashed. In case of extended illness beyond 12 days, employees may apply for medical leave without pay, subject to HR approval.', 'source': 'HR Policy'}, {'text': 'Casual Leave:\nEmployees are entitled to 6 days of casual leave per year. Casua